In [11]:
import os
import pandas as pd
import json
import re

root_dir = os.getcwd()
csv_file_path = os.path.join(root_dir, "data/老现西第一册.csv")
df = pd.read_csv(csv_file_path, encoding="utf-8")

In [6]:
def saveFile(data, data_file):
    with open(data_file, mode="w", encoding="utf-8") as json_file:
        json.dump(data, json_file, ensure_ascii=False, indent=4)


def toWordJson(filter_df, WORD_BOOK, data_file, dataInBook_file):
    data = []
    dataInBook = []

    for index, row in filter_df.iterrows():
        clean_word = re.sub(r"[^\w\s]", "", row["translation"]).split("\n")[0]

        # todo 命名规则需要符合url规范
        id = f"{index}_{clean_word}"
        voiceUrl = f"modernSpanish/{WORD_BOOK}/{id}.mp3"

        # 移动mp3文件
        # source_voice_path = os.path.join(
        #     root_dir, "data", "modernSpanish2", f"{id}.mp3"
        # )
        # dest_voice_path = os.path.join(root_dir, "data", WORD_BOOK, f"{id}.mp3")
        # if os.path.exists(source_voice_path):
        #     os.rename(source_voice_path, dest_voice_path)

        entry = {
            "id": id,
            "pos": row["pos"] if "pos" in row and pd.notna(row["pos"]) else None,
            "word": row["word"],
            "definition": (
                row["definition"]
                if "definition" in row and pd.notna(row["definition"])
                else ""
            ),
            "translation": (
                f"{row['pos']} {row['translation']}"
                if "pos" in row and pd.notna(row["pos"])
                else row["translation"]
            ),
            "voiceUrl": voiceUrl,
        }
        data.append(entry)
        dataInBook.append({"wb_id": WORD_BOOK, "word_id": id})

    saveFile(data, data_file)
    saveFile(dataInBook, dataInBook_file)
    return len(data)

In [12]:
# Get unique volumes and sort them
volumes = df["volume"].unique()
volumes.sort()
volumes

all_books = []
for v in volumes:
    WORD_BOOK = f"modernSpanish_{v}"
    os.makedirs(os.path.join(root_dir, "data/modernSpanish", WORD_BOOK), exist_ok=True)

    data_file = f"{root_dir}/data/modernSpanish/{WORD_BOOK}/{WORD_BOOK}.json"
    dataInBook_file = (
        f"{root_dir}/data/modernSpanish/{WORD_BOOK}/{WORD_BOOK}_inBook.json"
    )
    print(data_file)

    filtered_df = df[df["volume"] == v]
    total = toWordJson(filtered_df, WORD_BOOK, data_file, dataInBook_file)

    entry = {
        "id": WORD_BOOK,
        "name": f"现代西班牙语 {v}",
        "description": f"现代西班牙语 {v}",
        "total": total,
        "tag": "",
        "cover": "https://spanish-hhw.oss-cn-shanghai.aliyuncs.com/system/ModernSpanish1.png",
    }
    all_books.append(entry)

saveFile(all_books, f"{root_dir}/data/wordBooks.json")

/Users/heavenmei/Desktop/助管/AI-Spanish/ai-spanish/spanish-words/data/modernSpanish/modernSpanish_101/modernSpanish_101.json
/Users/heavenmei/Desktop/助管/AI-Spanish/ai-spanish/spanish-words/data/modernSpanish/modernSpanish_102/modernSpanish_102.json
/Users/heavenmei/Desktop/助管/AI-Spanish/ai-spanish/spanish-words/data/modernSpanish/modernSpanish_103/modernSpanish_103.json
/Users/heavenmei/Desktop/助管/AI-Spanish/ai-spanish/spanish-words/data/modernSpanish/modernSpanish_104/modernSpanish_104.json
/Users/heavenmei/Desktop/助管/AI-Spanish/ai-spanish/spanish-words/data/modernSpanish/modernSpanish_105/modernSpanish_105.json
/Users/heavenmei/Desktop/助管/AI-Spanish/ai-spanish/spanish-words/data/modernSpanish/modernSpanish_106/modernSpanish_106.json
/Users/heavenmei/Desktop/助管/AI-Spanish/ai-spanish/spanish-words/data/modernSpanish/modernSpanish_107/modernSpanish_107.json
/Users/heavenmei/Desktop/助管/AI-Spanish/ai-spanish/spanish-words/data/modernSpanish/modernSpanish_108/modernSpanish_108.json
/Users/h

## Merge Words JSON

In [9]:
def merge_json_files(root_dir):
    merged_data = []
    for subdir, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith(".json") and "_inBook" not in file:
                file_path = os.path.join(subdir, file)

                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    merged_data += data
    return merged_data


# Merge JSON files in 'data' directory
merged_json_data = merge_json_files(os.path.join(root_dir, "data/modernSpanish"))
saveFile(merged_json_data, os.path.join(root_dir, "data", "modernSpanish.json"))